# Dashboard Discovery & Inventory

Discovers Lakeview dashboards using **system tables** for instant catalog filtering.

**Serverless-compatible:** Uses Volumes for all file operations.

**Usage:**
- Test: `SOURCE_CATALOG_FILTER = "catalog_name"`, `TOP_N = 5` (⚡ ~10-30s)
- Full inventory: `SOURCE_CATALOG_FILTER = None`, `TOP_N = None` (⚡ ~30-60s)

**Performance:** System tables query = instant results even with 40K+ dashboards!

**Output:** JSONs and CSV in Volumes

## 1. Install & Import

In [ ]:
%pip install databricks-sdk pandas --quiet
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import DashboardView
import pandas as pd
import json, re, os, csv
from datetime import datetime

print("✅ Libraries imported")

## 2. Configuration

In [ ]:
# Workspace ID (from URL: ?o=WORKSPACE_ID)
WORKSPACE_ID = 6005387719195814

# Cross-workspace (leave as None for same workspace)
SOURCE_WORKSPACE_URL = None
SOURCE_WORKSPACE_TOKEN = None

# Catalog filter (None = all dashboards)
SOURCE_CATALOG_FILTER = "archana_krish_fe_dsa"

# Number of dashboards (None = all)
TOP_N = 5

# Output paths (use Volumes for serverless compatibility)
OUTPUT_VOLUME = "/Volumes/archana_krish_fe_dsa/default/dashboard_migration"  # UPDATE THIS
OUTPUT_PATH = f"{OUTPUT_VOLUME}/jsons/"
CSV_OUTPUT_PATH = f"{OUTPUT_VOLUME}/dashboard_inventory.csv"

print(f"✅ Config: {SOURCE_CATALOG_FILTER or 'ALL'} catalogs, TOP_N={TOP_N or 'ALL'}")
print(f"   Output: {OUTPUT_VOLUME}")

## 3. Discover Dashboards

In [ ]:
# Initialize client
if SOURCE_WORKSPACE_URL and SOURCE_WORKSPACE_TOKEN:
    client = WorkspaceClient(host=SOURCE_WORKSPACE_URL, token=SOURCE_WORKSPACE_TOKEN)
    print("🔗 Cross-workspace mode")
else:
    client = WorkspaceClient()
    print("✅ Notebook-native auth")

# Fast catalog filtering using system tables (avoids 40K+ API calls)
if SOURCE_CATALOG_FILTER:
    print(f"🔍 Using system tables to find dashboards with catalog: {SOURCE_CATALOG_FILTER}")
    
    # Query system tables - returns results in seconds, not hours!
    dashboard_ids_df = spark.sql(f"""
        SELECT DISTINCT entity_id as dashboard_id
        FROM system.access.table_lineage
        WHERE workspace_id = {WORKSPACE_ID}
            AND entity_type = 'DASHBOARD_V3'
            AND source_table_catalog = '{SOURCE_CATALOG_FILTER}'
    """)
    
    dashboard_ids = [row.dashboard_id for row in dashboard_ids_df.collect()]
    print(f"✅ Found {len(dashboard_ids)} dashboards using catalog '{SOURCE_CATALOG_FILTER}'")
    
    # Apply TOP_N limit before fetching metadata
    dashboard_ids = dashboard_ids[:TOP_N] if TOP_N else dashboard_ids
    
    # Now fetch metadata only for matching dashboards
    inventory_dashboards = []
    for dash_id in dashboard_ids:
        try:
            full = client.lakeview.get(dash_id)
            inventory_dashboards.append({
                'dashboard_id': dash_id,
                'name': full.display_name or f"Dashboard_{dash_id}",
                'path': full.path or f"/sql/dashboards/{dash_id}",
                'warehouse_id': full.warehouse_id
            })
        except Exception as e:
            print(f"  ⚠️  {dash_id}: {e}")
    
else:
    # No filter - list all dashboards (fast with BASIC view)
    print("📋 Listing all dashboards (no catalog filter)")
    all_dashboards_raw = list(client.lakeview.list(
        page_size=100,
        view=DashboardView.DASHBOARD_VIEW_BASIC,
        show_trashed=False
    ))
    print(f"✅ Found {len(all_dashboards_raw)} dashboards")
    
    # Apply TOP_N limit
    limited = all_dashboards_raw[:TOP_N] if TOP_N else all_dashboards_raw
    
    inventory_dashboards = [{
        'dashboard_id': d.dashboard_id,
        'name': d.display_name or f"Dashboard_{d.dashboard_id}",
        'path': getattr(d, 'path', f"/sql/dashboards/{d.dashboard_id}"),
        'warehouse_id': getattr(d, 'warehouse_id', None)
    } for d in limited]

print(f"\n📋 Selected {len(inventory_dashboards)} dashboards\n")
if inventory_dashboards:
    display(pd.DataFrame(inventory_dashboards))

## 4. Enrich Metadata

In [ ]:
dashboards = []

for dash_info in inventory_dashboards:
    dashboard_id = dash_info['dashboard_id']
    print(f"📊 {dash_info['name'][:50]}")
    
    try:
        full = client.lakeview.get(dashboard_id)
        
        # Get published status
        published = False
        try:
            client.lakeview.get_published(dashboard_id)
            published = True
        except:
            pass
        
        # Get table lineage
        try:
            lineage = spark.sql(f"""
                SELECT 
                    COUNT(DISTINCT source_table_catalog) as catalog_count,
                    COUNT(DISTINCT source_table_name) as table_count
                FROM system.access.table_lineage
                WHERE workspace_id = {WORKSPACE_ID}
                    AND entity_id = '{dashboard_id}'
                    AND entity_type = 'DASHBOARD_V3'
            """).collect()[0]
            catalog_count = lineage.catalog_count or 0
            table_count = lineage.table_count or 0
        except:
            catalog_count = 0
            table_count = 0
        
        dashboards.append({
            'dashboard_id': dashboard_id,
            'name': full.display_name,
            'path': full.path,
            'published': published,
            'catalog_count': catalog_count,
            'table_count': table_count,
            'owner': full.create_time.strftime('%Y-%m-%d') if full.create_time and hasattr(full.create_time, 'strftime') else 'Unknown',
            'link': f"/sql/dashboards/{dashboard_id}"
        })
        print(f"  ✅ Published={published}, Tables={table_count}\n")
    except Exception as e:
        print(f"  ⚠️  {e}\n")

print(f"✅ Enriched {len(dashboards)} dashboards\n")
if dashboards:
    display(pd.DataFrame(dashboards))

## 5. Export JSONs

In [ ]:
# Create volume directory using dbutils (serverless-compatible)
try:
    dbutils.fs.mkdirs(OUTPUT_PATH)
    print(f"✅ Created directory: {OUTPUT_PATH}")
except Exception as e:
    print(f"ℹ Directory exists or created: {OUTPUT_PATH}")

exported = 0

for dash in dashboards:
    try:
        full = client.lakeview.get(dash['dashboard_id'])
        content = full.serialized_dashboard
        
        if content:
            # Write to Volumes (serverless-compatible)
            output_file = f"{OUTPUT_PATH}{dash['dashboard_id']}.lvdash.json"
            dbutils.fs.put(output_file, content, overwrite=True)
            exported += 1
            print(f"✅ {dash['name'][:40]} ({len(content)/1024:.1f} KB)")
    except Exception as e:
        print(f"⚠️  {dash['name'][:40]}: {e}")

print(f"\n✅ Exported {exported}/{len(dashboards)} dashboards to {OUTPUT_PATH}")

## 6. Export CSV & Output

In [ ]:
# Export to CSV
# Export to CSV using pandas (serverless-compatible)
try:
    df_export = pd.DataFrame(dashboards)
    
    # Convert to Spark DataFrame and write to Volumes
    spark_df = spark.createDataFrame(df_export)
    
    # Write as single CSV file
    temp_path = f"{OUTPUT_VOLUME}/temp_csv"
    spark_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv(temp_path)
    
    # Find the CSV file (Spark creates part files)
    csv_files = [f.path for f in dbutils.fs.ls(temp_path) if f.path.endswith('.csv')]
    if csv_files:
        dbutils.fs.mv(csv_files[0], CSV_OUTPUT_PATH)
        dbutils.fs.rm(temp_path, recurse=True)
    
    print(f"✅ CSV: {CSV_OUTPUT_PATH}")
except Exception as e:
    print(f"⚠️  CSV export failed: {e}")
    print(f"   Dashboards available in memory as 'dashboards' variable")

# Generate Python list
print("\n" + "="*80)
print("📋 DASHBOARD IDs FOR MIGRATION")
print("="*80)
print("\nDASHBOARD_IDS = [")
for d in dashboards:
    print(f"    '{d['dashboard_id']}',  # {d['name']}")
print("]")

print(f"\n✅ Complete: {len(dashboards)} dashboards ready for migration")
print(f"📂 JSONs: {OUTPUT_PATH}")
print(f"📄 CSV: {CSV_OUTPUT_PATH}")